# 01_prepare_vggface2 — shared external storage

목적: VGGFace2 공식 train 목록과 제공 좌표 기반 정렬 이미지를 부모 학습용으로 준비합니다.

UCFace explicit settings take priority; otherwise FaceX-Zoo conventional training.
Target-only FP32 zero-radicand safeguard; 112×112 inputs; full training lists and final batches.
VGGFace2: provided five-point landmarks + original FaceX-Zoo cropper.
TinyFace/SurvFace/SCface: user-approved resize-only, default bicubic; no face re-detection.
Existing failed-condition checkpoints are preserved as records. 00 saves settings; 01–04 start with EXECUTE=False.


## 1. 설정

선택지·범위가 적힌 변수만 필요한 조건으로 조정합니다. 공통 변경은 00과 같은 SETTINGS_PATH를 사용하는 후속 단계에 일관되게 반영합니다.


In [ ]:
# 목적: 00의 설정을 읽고 VGGFace2 압축 해제·목록 생성·전처리의 실행 범위를 정합니다.

from pathlib import Path
import sys, json, os
BASE_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "fine-tune/modules").is_dir())  # 저장소 위치 자동 탐색값입니다. 직접 변경하지 않습니다.
DATASET_ROOT = Path(os.environ.get("FACE_DATASET_ROOT", "F:/face_datasets" if os.name == "nt" else "/data/face_datasets"))  # 공유 데이터 루트; 환경변수 FACE_DATASET_ROOT 또는 이 Path를 실제 경로로 지정합니다.
MODEL_ROOT = Path(os.environ.get("FACE_MODEL_ROOT", "F:/models" if os.name == "nt" else "/data/models"))  # 공유 모델 루트; 환경변수 FACE_MODEL_ROOT 또는 이 Path를 실제 경로로 지정합니다.
SETTINGS_PATH = DATASET_ROOT / ".state/facex/shared_v1_boundary/notebook_settings_mobilefacenet.json"  # 00에서 저장하고 01~04에서 함께 읽을 JSON입니다. backbone/실험별로 파일을 구분합니다.
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
# 00의 공통값을 읽습니다. 아래 변수 변경이 저장값/YAML보다 우선합니다.
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])  # 중복 없는 정수 목록(0 <= seed < 2**32); 일부 seed만 선택 가능하며 후속 단계도 동일하게 맞춥니다.
HEADS = list(saved["campaign"]["heads"])  # "arcface", "adaface", "magface", "cosface" 중 중복 없이 선택; 02 부모는 ArcFace로 고정입니다.
# 원본 데이터·공유 전처리·부모·과거 실패 기록은 이 보존 옵션의 삭제 대상이 아닙니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]  # True=이번 학습의 주기별 checkpoint 보존, False=학습 중 오래된 중간 checkpoint 정리.
NUM_WORKERS = 0  # 0~12 정수; 이 셀의 Windows 노트북 직접 batch 검사에는 0을 권장합니다.
DEVICE = saved["project"]["device"]  # "cuda:0" 또는 사용 가능한 GPU 번호("cuda:1" 등); 정식 학습은 CUDA 필수이며 CPU fallback은 없습니다.
DATASET = dict(saved["dataset"])  # VGGFace2 경로·입력 설정입니다. 112×112와 pixel_mean=127.5/pixel_std=128.0 조건을 유지합니다.
BACKBONE = dict(saved["backbone"])  # 00에서 MobileFaceNet/Resnet IR-50 선택; 1024D·dropout 0.6·out_h/out_w=7 조건을 유지합니다.
# 정식 fine-tune: optimizer=adam, lr=0.001, epoches=40, milestones=[10,20,30], lr_decay_ratio=0.1 고정.
# batch_size는 MobileFaceNet=64, Resnet=2 이상(00 기본 16); save_freq는 1 이상 epoch, drop_last=False 유지.
TRAINING = dict(saved["training"])  # 학습/저장 설정 사전입니다. 위 고정 학습 조건과 조절 가능한 저장 간격을 구분합니다.
HEAD_PARAMETERS = saved["heads"]  # head별 margin/scale 기준값입니다. 정식 fine-tune은 이를 검증하므로 head 선택은 HEADS에서 합니다.
PARENT_HEAD = dict(saved["parent_head"])  # 부모는 ArcFace, margin_arc=0.35/margin_am=0/scale=32로 고정합니다.
# flip_probability/affine_probability: 0~1, rotation_degrees: 0 이상, scale_range=[low,high]: 0<low<=high.
# border_mode="reflect"/"constant", center="image_center" 고정; 현재 수치를 UCFace 명시값으로 간주하지 않습니다.
AUGMENTATION = dict(saved["augmentation"])  # 증강 수치 변경은 별도 학습 조건입니다. 새 namespace와 부모 학습으로 구분합니다.
ALIGNMENT = dict(saved["alignment"])  # VGGFace2=official_landmarks, 저해상도=resize_only; 보간법은 bicubic/bilinear 중 선택합니다.
PATHS = dict(saved["paths"])  # checkpoint/export/log/report/cache 경로; 00에서 namespace별로 정하고 후속 단계가 이어받습니다.
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]  # True=동일 조건의 검증된 완료 결과 재사용; False=완료 경로가 있으면 중단하며 덮어쓰지 않습니다.
EXECUTE = False  # False=계획 확인, True=목록 생성과 선택한 압축 해제/전처리·완료 입력 검사를 수행합니다.
EXTRACT_TRAIN_ARCHIVE = False  # False=기존 추출 폴더 사용, True=로컬 archive 해제; EXECUTE=True일 때만 실행합니다.
VAL_EVERY = None  # 정식 경로는 None(공식 train 전체); 2 이상 정수의 legacy 행 단위 분할은 별도 점검용입니다.
RUN_PREPROCESSING = False  # False=기존 완료 전처리 사용, True=전처리 실행; EXECUTE=True와 함께 사용합니다.
PREPROCESS_MAX_IMAGES = 32  # None=선택 학습 이미지 전체, 양의 정수=부분 점검 수; 부분 결과는 정식 학습에 사용하지 않습니다.
PREPROCESS_DEVICE = "cpu"  # "cpu"만 지원합니다. 좌표 기반 정렬/resize 처리이며 검출 모델용 GPU 설정이 아닙니다.
# 현재 노트북 변수 -> 실제 모듈 설정. 입력 112x112 이외는 현 구조에서 명시적 중단.
# 위 변수들을 실제 모듈 설정으로 묶어 검증합니다. 아래 전달부는 직접 편집할 필요가 없습니다.
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
    "project": {"seed": SEEDS[0], "seed_list": SEEDS, "keep_raw_results": KEEP_RAW_RESULTS, "device": DEVICE},
    "campaign": {"heads": HEADS, "reuse_completed": REUSE_COMPLETED},
    "dataset": DATASET, "backbone": BACKBONE, "heads": HEAD_PARAMETERS,
    "training": {**TRAINING, "num_workers": NUM_WORKERS}, "paths": PATHS, "parent_head":PARENT_HEAD, "augmentation":AUGMENTATION, "alignment":ALIGNMENT,
})


## 2. 경로 확인
압축 해제는 원본 JPEG를 `DATASET["raw_root"]/train/` 아래에 저장합니다. 얼굴 정렬이나 리사이즈를 수행하는 단계는 아닙니다. 로컬 압축파일을 사용하며 다운로드하지 않습니다.

In [ ]:
# 목적: 원본 archive와 추출 목적지, 실행 스위치를 확인합니다.

print(DATASET)
print("EXECUTE:", EXECUTE, "VAL_EVERY:", VAL_EVERY)
print("EXTRACT_TRAIN_ARCHIVE:", EXTRACT_TRAIN_ARCHIVE)
print("extract:", DATASET["train_archive"], "->", DATASET["raw_root"])

## 3. 학습 이미지 압축 해제 및 목록 생성
최초 실행은 전체 압축 해제에 시간이 걸립니다. 약 10초마다 처리한 파일 수를 표시합니다. 중단 후 다시 실행하면 압축파일을 처음부터 읽되, 기존 파일은 내용이 동일한지 확인하고 없는 파일만 추가합니다. 기존 파일 내용이 다르면 덮어쓰지 않고 중단합니다.

압축 해제를 이미 완료했다면 첫 셀에서 `EXTRACT_TRAIN_ARCHIVE=False`로 설정할 수 있습니다. 기존 train/val 목록은 그대로 검증·재사용합니다.

In [ ]:
# 목적: 실행 시 필요하면 archive를 해제하고 공식 training 이미지 목록을 생성·검증합니다.

from modules.archive_preparation import extract_train_archive
from modules.notebook_workflow import prepare_indices
extraction_result = None
if EXECUTE:
    if EXTRACT_TRAIN_ARCHIVE:
        extraction_result = extract_train_archive(DATASET["train_archive"], DATASET["raw_root"])
    split_result = prepare_indices(settings, VAL_EVERY)
    print(split_result)
else:
    split_result = None
    print("계획 모드: 압축 해제 및 목록 생성 미실행")

## 4. 실제 이미지 전처리 확인

In [ ]:
# 목적: 선택한 범위의 정렬을 수행하고, 전체 정렬 완료 기록이 있으면 학습 입력 batch를 확인합니다.

from modules.process_runner import launch_alignment
from modules.alignment import bind_aligned_dataset
from modules.notebook_workflow import make_loaders
if EXECUTE and RUN_PREPROCESSING:
    launch_alignment(BASE_DIR, settings, "vggface2", max_images=PREPROCESS_MAX_IMAGES, device=PREPROCESS_DEVICE)
if EXECUTE and Path(settings["dataset"]["alignment_report"]).is_file():
    aligned_settings = bind_aligned_dataset(settings)
    train_loader, val_loader = make_loaders(aligned_settings)
    images, labels = next(iter(train_loader))
    print(tuple(images.shape), "validation:", val_loader)
    del train_loader
else:
    print("Full alignment is required before 02. A bounded audit is not training-ready.")


## 5. 결과 해석

In [ ]:
# 목적: 공식 train 전체 사용과 오류 검토 원칙을 확인해 부분 점검을 학습 준비 완료와 구분합니다.

print("Official VGGFace2 training identities only; no 95:5 split and no test identities added.")
print("Review missing coordinates/decode errors. No automatic sample deletion or fallback.")
print("Existing source data and shared_parent_v3 results are preserved.")
